        # Fake News Detection Training and Evaluation

        This notebook reproduces the backend preprocessing and compares three models:

        1. Logistic Regression with TF-IDF
        2. Naive Bayes with TF-IDF
        3. Random Forest with engineered metadata features

        **Outputs**
        - 5-fold cross-validation summaries
        - ROC curve comparison
        - Confusion matrix comparison
        - Classification report tables
        - Top feature importance plot for the best model
        - Metrics exported to `notebooks/results.json`
        


In [ ]:
        from __future__ import annotations

        import json
        import sys
        from pathlib import Path

        import matplotlib.pyplot as plt
        import numpy as np
        import pandas as pd
        import seaborn as sns
        from IPython.display import display
        from sklearn.base import clone
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import (
            ConfusionMatrixDisplay,
            accuracy_score,
            classification_report,
            confusion_matrix,
            f1_score,
            precision_score,
            recall_score,
            roc_auc_score,
            roc_curve,
        )
        from sklearn.model_selection import StratifiedKFold
        from sklearn.naive_bayes import MultinomialNB

        CURRENT_DIR = Path.cwd().resolve()
        if (CURRENT_DIR / "backend").exists():
            PROJECT_ROOT = CURRENT_DIR
            NOTEBOOK_DIR = CURRENT_DIR / "notebooks"
        else:
            NOTEBOOK_DIR = CURRENT_DIR
            PROJECT_ROOT = CURRENT_DIR.parent

        BACKEND_DIR = PROJECT_ROOT / "backend"
        FIGURES_DIR = NOTEBOOK_DIR / "figures"
        FIGURES_DIR.mkdir(parents=True, exist_ok=True)
        RESULTS_PATH = NOTEBOOK_DIR / "results.json"

        if str(BACKEND_DIR) not in sys.path:
            sys.path.insert(0, str(BACKEND_DIR))

        from data_utils import LIAR_COLUMNS, preprocess_liar_dataframe, resolve_data_directory
        from features import (
            NUMERIC_FEATURE_NAMES,
            build_numeric_feature_matrix,
            build_party_encoder,
            build_speaker_lookup,
            build_subject_vectorizer,
            build_tfidf_vectorizer,
            combine_engineered_features,
        )

        sns.set_theme(style="whitegrid", context="talk")
        plt.rcParams.update(
            {
                "figure.dpi": 120,
                "savefig.dpi": 300,
                "figure.facecolor": "white",
                "axes.facecolor": "white",
                "axes.titlesize": 18,
                "axes.labelsize": 14,
                "legend.fontsize": 11,
                "xtick.labelsize": 11,
                "ytick.labelsize": 11,
            }
        )
        


In [ ]:
        def save_figure(filename: str) -> None:
            plt.tight_layout()
            plt.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
            plt.show()


        def load_split(split_name: str, data_dir: Path) -> pd.DataFrame:
            split_path = data_dir / f"{split_name}.tsv"
            raw_frame = pd.read_csv(split_path, sep="\t", header=None, names=LIAR_COLUMNS)
            return preprocess_liar_dataframe(raw_frame)


        def baseline_model_factory(model_name: str):
            if model_name == "LogisticRegression":
                return LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=42,
                )
            if model_name == "NaiveBayes":
                return MultinomialNB(alpha=0.5)
            raise ValueError(f"Unsupported baseline model: {model_name}")


        def random_forest_factory():
            return RandomForestClassifier(
                n_estimators=300,
                max_depth=30,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42,
            )


        def prepare_baseline_features(train_frame: pd.DataFrame, eval_frame: pd.DataFrame):
            tfidf = build_tfidf_vectorizer()
            X_train = tfidf.fit_transform(train_frame["statement"])
            X_eval = tfidf.transform(eval_frame["statement"])
            return X_train, X_eval, tfidf


        def prepare_engineered_features(train_frame: pd.DataFrame, eval_frame: pd.DataFrame):
            speaker_scores, default_speaker_score = build_speaker_lookup(train_frame)

            tfidf = build_tfidf_vectorizer()
            X_train_text = tfidf.fit_transform(train_frame["statement"])
            X_eval_text = tfidf.transform(eval_frame["statement"])

            party_encoder = build_party_encoder()
            X_train_party = party_encoder.fit_transform(train_frame[["party"]])
            X_eval_party = party_encoder.transform(eval_frame[["party"]])

            subject_vectorizer = build_subject_vectorizer()
            X_train_subject = subject_vectorizer.fit_transform(train_frame["subject"])
            X_eval_subject = subject_vectorizer.transform(eval_frame["subject"])

            X_train_numeric = build_numeric_feature_matrix(train_frame, speaker_scores, default_speaker_score)
            X_eval_numeric = build_numeric_feature_matrix(eval_frame, speaker_scores, default_speaker_score)

            X_train = combine_engineered_features(
                X_train_text,
                X_train_numeric,
                X_train_party,
                X_train_subject,
            )
            X_eval = combine_engineered_features(
                X_eval_text,
                X_eval_numeric,
                X_eval_party,
                X_eval_subject,
            )
            return {
                "X_train": X_train,
                "X_eval": X_eval,
                "tfidf": tfidf,
                "text_feature_names": tfidf.get_feature_names_out().tolist(),
                "speaker_scores": speaker_scores,
                "default_speaker_score": default_speaker_score,
            }


        def compute_metrics(y_true, y_pred, y_prob):
            return {
                "accuracy": float(accuracy_score(y_true, y_pred)),
                "precision": float(precision_score(y_true, y_pred, zero_division=0)),
                "recall": float(recall_score(y_true, y_pred, zero_division=0)),
                "f1_score": float(f1_score(y_true, y_pred, zero_division=0)),
                "roc_auc": float(roc_auc_score(y_true, y_prob)),
                "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
            }
        


In [ ]:
        data_dir = resolve_data_directory(PROJECT_ROOT / "data" if (PROJECT_ROOT / "data").exists() else None)

        train_df = load_split("train", data_dir)
        valid_df = load_split("valid", data_dir)
        test_df = load_split("test", data_dir)
        train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)

        y_train_valid = train_valid_df["label"].to_numpy(dtype=int)
        y_test = test_df["label"].to_numpy(dtype=int)

        pd.DataFrame(
            {
                "split": ["train", "valid", "train_valid", "test"],
                "rows": [len(train_df), len(valid_df), len(train_valid_df), len(test_df)],
            }
        )
        


        ## 5-Fold Cross-Validation

        Cross-validation is performed with fold-specific preprocessing so the TF-IDF and metadata encoders are refit inside each fold, matching production behavior without leakage.
        


In [ ]:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        model_names = ["LogisticRegression", "NaiveBayes", "RandomForest"]
        cv_metrics = {model_name: [] for model_name in model_names}

        for fold_index, (train_idx, valid_idx) in enumerate(cv.split(train_valid_df, y_train_valid), start=1):
            fold_train = train_valid_df.iloc[train_idx].reset_index(drop=True)
            fold_valid = train_valid_df.iloc[valid_idx].reset_index(drop=True)
            y_fold_train = fold_train["label"].to_numpy(dtype=int)
            y_fold_valid = fold_valid["label"].to_numpy(dtype=int)

            for model_name in model_names:
                if model_name in {"LogisticRegression", "NaiveBayes"}:
                    X_fold_train, X_fold_valid, _ = prepare_baseline_features(fold_train, fold_valid)
                    model = baseline_model_factory(model_name)
                else:
                    engineered_bundle = prepare_engineered_features(fold_train, fold_valid)
                    X_fold_train = engineered_bundle["X_train"]
                    X_fold_valid = engineered_bundle["X_eval"]
                    model = random_forest_factory()

                model.fit(X_fold_train, y_fold_train)
                fold_pred = model.predict(X_fold_valid)
                fold_prob = model.predict_proba(X_fold_valid)[:, 1]

                fold_metrics = compute_metrics(y_fold_valid, fold_pred, fold_prob)
                fold_metrics["fold"] = fold_index
                cv_metrics[model_name].append(fold_metrics)

        cv_summary = []
        for model_name, scores in cv_metrics.items():
            score_frame = pd.DataFrame(scores)
            cv_summary.append(
                {
                    "model": model_name,
                    "cv_accuracy_mean": score_frame["accuracy"].mean(),
                    "cv_accuracy_std": score_frame["accuracy"].std(),
                    "cv_f1_mean": score_frame["f1_score"].mean(),
                    "cv_f1_std": score_frame["f1_score"].std(),
                    "cv_roc_auc_mean": score_frame["roc_auc"].mean(),
                    "cv_roc_auc_std": score_frame["roc_auc"].std(),
                }
            )

        cv_summary_df = pd.DataFrame(cv_summary).sort_values("cv_f1_mean", ascending=False).reset_index(drop=True)
        display(cv_summary_df)
        


        ## Final Training on Train + Validation, Evaluation on Test
        


In [ ]:
        final_models = {}
        evaluation_results = {}
        classification_reports = {}
        roc_curves = {}

        for model_name in model_names:
            if model_name in {"LogisticRegression", "NaiveBayes"}:
                X_train_final, X_test_final, tfidf = prepare_baseline_features(train_valid_df, test_df)
                model = baseline_model_factory(model_name)
                artifact = {
                    "model": model,
                    "tfidf": tfidf,
                    "text_feature_names": tfidf.get_feature_names_out().tolist(),
                }
            else:
                engineered_bundle = prepare_engineered_features(train_valid_df, test_df)
                X_train_final = engineered_bundle["X_train"]
                X_test_final = engineered_bundle["X_eval"]
                model = random_forest_factory()
                artifact = {
                    "model": model,
                    "tfidf": engineered_bundle["tfidf"],
                    "text_feature_names": engineered_bundle["text_feature_names"],
                }

            model.fit(X_train_final, y_train_valid)
            test_pred = model.predict(X_test_final)
            test_prob = model.predict_proba(X_test_final)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, test_prob)

            final_models[model_name] = artifact
            evaluation_results[model_name] = compute_metrics(y_test, test_pred, test_prob)
            classification_reports[model_name] = classification_report(
                y_test,
                test_pred,
                target_names=["FAKE", "REAL"],
                output_dict=True,
                zero_division=0,
            )
            roc_curves[model_name] = {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist(),
            }
        


        ## Classification Reports
        


In [ ]:
        report_tables = []
        for model_name, report in classification_reports.items():
            report_frame = pd.DataFrame(report).transpose().reset_index().rename(columns={"index": "metric"})
            report_frame.insert(0, "model", model_name)
            report_tables.append(report_frame)

        classification_report_df = pd.concat(report_tables, ignore_index=True)
        display(classification_report_df)
        


        ## ROC Curves
        


In [ ]:
        fig, ax = plt.subplots(figsize=(10, 8))
        palette = {
            "LogisticRegression": "#1f77b4",
            "NaiveBayes": "#ff7f0e",
            "RandomForest": "#2ca02c",
        }

        for model_name, curve in roc_curves.items():
            ax.plot(
                curve["fpr"],
                curve["tpr"],
                linewidth=2.5,
                label=f"{model_name} (AUC={evaluation_results[model_name]['roc_auc']:.3f})",
                color=palette[model_name],
            )

        ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
        ax.set_title("ROC Curves on Test Split")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right")

        save_figure("train_eval_roc_curves.png")
        


        ## Confusion Matrices
        


In [ ]:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        for axis, model_name in zip(axes, model_names):
            matrix = np.array(evaluation_results[model_name]["confusion_matrix"])
            display_obj = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=["FAKE", "REAL"])
            display_obj.plot(ax=axis, cmap="YlOrRd", colorbar=False)
            axis.set_title(model_name)

        save_figure("train_eval_confusion_matrices.png")
        


        ## Feature Importance for the Best Model

        The best model is selected by mean cross-validation F1 score. The chart below ranks the top 20 TF-IDF features only.
        


In [ ]:
        best_model_name = cv_summary_df.iloc[0]["model"]
        best_artifact = final_models[best_model_name]
        feature_names = best_artifact["text_feature_names"]
        fitted_model = best_artifact["model"]

        if best_model_name == "RandomForest":
            importance_values = fitted_model.feature_importances_[: len(feature_names)]
        elif best_model_name == "LogisticRegression":
            importance_values = np.abs(fitted_model.coef_[0][: len(feature_names)])
        else:
            importance_values = np.abs(
                (fitted_model.feature_log_prob_[1] - fitted_model.feature_log_prob_[0])[: len(feature_names)]
            )

        top_indices = np.argsort(importance_values)[-20:][::-1]
        top_features = pd.DataFrame(
            {
                "feature": np.array(feature_names)[top_indices],
                "importance": importance_values[top_indices],
            }
        )

        fig, ax = plt.subplots(figsize=(12, 8))
        sns.barplot(data=top_features, x="importance", y="feature", palette="viridis", ax=ax)
        ax.set_title(f"Top 20 TF-IDF Features for {best_model_name}")
        ax.set_xlabel("Importance")
        ax.set_ylabel("TF-IDF feature")

        save_figure("train_eval_top_feature_importance.png")
        top_features
        


        ## Persist Metrics
        


In [ ]:
        results_payload = {
            "best_model": best_model_name,
            "cross_validation": {
                model_name: metrics
                for model_name, metrics in cv_metrics.items()
            },
            "cross_validation_summary": cv_summary_df.to_dict(orient="records"),
            "test_metrics": evaluation_results,
            "classification_reports": classification_reports,
            "figure_paths": {
                "roc_curves": str(FIGURES_DIR / "train_eval_roc_curves.png"),
                "confusion_matrices": str(FIGURES_DIR / "train_eval_confusion_matrices.png"),
                "feature_importance": str(FIGURES_DIR / "train_eval_top_feature_importance.png"),
            },
        }

        RESULTS_PATH.write_text(json.dumps(results_payload, indent=2), encoding="utf-8")
        print(f"Saved metrics to {RESULTS_PATH}")
        
